# QPA Fidelity Decay — Unrolled Strategy (T=1) — IBM Boston

Based on `end_to_end_unrolled_pittsburgh_small.ipynb`, retargeted to IBM Boston.

**Configuration:**
- `N_RANDOM = 5000` Pauli twirling instances per λ
- `SHOTS = 3` shots per circuit instance
- λ sweep is **split into two passes** (`LAMBDA_PHASE`):
  - `"even"` → `0.0, 0.2, 0.4, 0.6, 0.8`
  - `"odd"`  → `0.1, 0.3, 0.5, 0.7, 0.9`
  - `"all"`  → `0.0, 0.1, …, 0.9`
- `N ∈ {3, 5, 7, 9, 11}`, `K = 2`, `T = 1`
- **Transpilation happens in [`find_best_transpilation_unrolled_boston_t1.ipynb`](find_best_transpilation_unrolled_boston_t1.ipynb)** and is loaded from disk here. Run that notebook first.
- Results are checkpointed to CSV + JSON after **every** λ point so the run can be resumed if interrupted

**Two-pass workflow.** Run once with `LAMBDA_PHASE = "even"`, then change it to `"odd"` and re-run the same notebook. The checkpoint files accumulate across phases — every cell automatically loads prior λ points from the CSV and only computes the missing ones, so the final plot/consolidation will show the full `0.0, 0.1, …, 0.9` set.

Pipeline:
1. Configuration
2. Backend setup (`ibm_boston`)
3. Circuit generation (unrolled)
4. **Load** pre-transpiled circuits from `find_best_transpilation_unrolled_boston_t1.ipynb` output
5. Noise application (Pauli twirling)
6. Submission to IBM, with per-λ checkpointing
7. Plot vs. theory (closed-form available for N=3, 5, 7 only)
8. Consolidated final save

In [1]:
import json
import os
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from qiskit import qpy
from qiskit.circuit import CircuitInstruction
from qiskit.circuit.library import XGate, RZGate
from qiskit_ibm_runtime import SamplerV2 as IBMSampler

from core.circuit_factory import CircuitFactory
from core.noise_models import PauliTwirlingStrategy
from execution.backend_handler import IBMRuntimeHandler
from analysis.result_processor import ResultProcessor

print("Imports loaded.")

Imports loaded.


## Step 1: Configuration

| Parameter | Value |
|-----------|-------|
| `N` | 3, 5, 7, 9, 11 |
| `K` | 2 |
| `T` | 1 |
| `LAMBDA_PHASE` | `"even"` / `"odd"` / `"all"` |
| λ values | union of `{0.0, 0.2, 0.4, 0.6, 0.8}` and `{0.1, 0.3, 0.5, 0.7, 0.9}` across passes |
| `N_RANDOM` | 5000 |
| `SHOTS` | 3 |
| `TRANSPILE_SEEDS` | 10 (lowest 2Q depth wins) |
| `DEVICE` | `ibm_boston` |

In [2]:
# ----- Experiment Parameters -----
K = 2                        # Qubits per register (d = 2^K = 4)
T = 1                        # Number of QPA trial rounds
N_RANDOM = 5000              # Pauli twirling instances per λ
SHOTS = 3                    # Shots per circuit instance
BATCH_SIZE = 1000            # Circuits per IBM job submission
TRANSPILE_SEEDS = 10         # Multi-seed transpilation, pick lowest 2Q depth
OPT_LEVEL = 3                # Transpiler optimization level
DEVICE = "ibm_boston"
NO_RESET = False

# ----- Lambda phase selector -----
# Pick which λ values to run *this pass*. The checkpoint/resume logic merges
# results across passes, so running "even" then "odd" gives the full set
# {0.0, 0.1, …, 0.9} in the consolidated CSV/JSON without recomputing.
#   "even" -> [0.0, 0.2, 0.4, 0.6, 0.8]
#   "odd"  -> [0.1, 0.3, 0.5, 0.7, 0.9]
#   "all"  -> [0.0, 0.1, …, 0.9]   (still skips any λ already in checkpoint)
LAMBDA_PHASE = "all"

LAMBDA_SETS = {
    "even": np.round(np.array([0.0, 0.2, 0.4, 0.6, 0.8]), 4),
    "odd":  np.round(np.array([0.1, 0.3, 0.5, 0.7, 0.9]), 4),
    "all":  np.round(np.arange(0.0, 1.0, 0.1), 4),
}
if LAMBDA_PHASE not in LAMBDA_SETS:
    raise ValueError(f"LAMBDA_PHASE must be one of {list(LAMBDA_SETS)}, got {LAMBDA_PHASE!r}")
lambdas = LAMBDA_SETS[LAMBDA_PHASE]

N_VALUES = [3, 5, 7, 9, 11]
EXPERIMENTS = [{"n": n, "k": K, "t": T, "label": f"N={n}, K={K}, T={T}"} for n in N_VALUES]

# Output directory for incremental checkpoints
RESULTS_DIR = os.path.join("data", "results", "end_to_end_unrolled_boston_t1")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"LAMBDA_PHASE = {LAMBDA_PHASE!r}")
print(f"Lambda sweep this pass ({len(lambdas)} points): {lambdas.tolist()}")
print(f"N_RANDOM={N_RANDOM}, SHOTS={SHOTS}, BATCH_SIZE={BATCH_SIZE}")
print(f"Transpile seeds per path: {TRANSPILE_SEEDS}")
print(f"Experiments: {[e['label'] for e in EXPERIMENTS]}")
print(f"Results directory: {RESULTS_DIR}")

LAMBDA_PHASE = 'all'
Lambda sweep this pass (10 points): [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
N_RANDOM=5000, SHOTS=3, BATCH_SIZE=1000
Transpile seeds per path: 10
Experiments: ['N=3, K=2, T=1', 'N=5, K=2, T=1', 'N=7, K=2, T=1', 'N=9, K=2, T=1', 'N=11, K=2, T=1']
Results directory: data/results/end_to_end_unrolled_boston_t1


## Step 2: Backend Setup

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(channel="ibm_quantum_platform", instance="crn:v1:bluemix:public:quantum-computing:us-east:a/6c63dae5281147f1a0449b36e0aaba3a:20efd56a-3db0-4aa6-9c69-2686147fcb87::", token="Makxjb7GBPH0H0uBGHzW8Cbsq1QCu4iMJfewolcT3Cp1", overwrite=True, set_as_default=True)
service = QiskitRuntimeService()
backend = service.backend(DEVICE)

print(f"Backend: {backend.name}")
print(f"Number of qubits: {backend.num_qubits}")
print(f"Backend status: {backend.status()}")

Backend: ibm_boston
Number of qubits: 156
Backend status: <qiskit_ibm_runtime.models.backend_status.BackendStatus object at 0x119077380>


## Step 3: Circuit Generation (Unrolled Strategy)

Number of unrolled paths per N (= 2^((N-1)/2) for T=1):
- N=3: 2 paths
- N=5: 4 paths
- N=7: 8 paths
- N=9: 16 paths
- N=11: 32 paths

In [4]:
golden_data_per_exp = {}

for exp in EXPERIMENTS:
    n = exp["n"]
    label = exp["label"]

    strategy = CircuitFactory.create_strategy("unrolled", K, T, n, no_reset=NO_RESET)
    strategy.set_noise_strategy(None)
    golden_data = strategy.build(0.0)

    golden_circuits = [item['circuit'] for item in golden_data]
    golden_metadata = [{k: v for k, v in item.items() if k != 'circuit'} for item in golden_data]

    golden_data_per_exp[n] = {
        'circuits': golden_circuits,
        'metadata': golden_metadata,
        'strategy': strategy,
    }

    print(f"\n--- {label} ---")
    print(f"  Total unrolled paths: {len(golden_circuits)}")
    print(f"  Qubits/circuit: {golden_circuits[0].num_qubits}, Clbits/circuit: {golden_circuits[0].num_clbits}")


--- N=3, K=2, T=1 ---
  Total unrolled paths: 2
  Qubits/circuit: 7, Clbits/circuit: 3

--- N=5, K=2, T=1 ---
  Total unrolled paths: 4
  Qubits/circuit: 12, Clbits/circuit: 4

--- N=7, K=2, T=1 ---
  Total unrolled paths: 8
  Qubits/circuit: 17, Clbits/circuit: 5

--- N=9, K=2, T=1 ---
  Total unrolled paths: 16
  Qubits/circuit: 22, Clbits/circuit: 6

--- N=11, K=2, T=1 ---
  Total unrolled paths: 32
  Qubits/circuit: 27, Clbits/circuit: 7


## Step 4: Load Pre-Transpiled Circuits

Best-of-N seed transpilation is done in [`find_best_transpilation_unrolled_boston_t1.ipynb`](find_best_transpilation_unrolled_boston_t1.ipynb), which writes one QPY file per path plus a `summary.json` to:

```
data/transpilations/end_to_end_unrolled_boston_t1/n{N}/
```

**Run that notebook first.** This cell loads the QPY files into `transpiled_per_exp[n]` and the per-path metadata (best seed, 2Q depth, gate counts, etc.) into `transpile_summary_per_exp[n]`, mirroring the structure the rest of this notebook expects. It errors loudly if any expected file is missing.

In [5]:
TRANSPILE_DIR = os.path.join("data", "transpilations", "end_to_end_unrolled_boston_t1")

if not os.path.isdir(TRANSPILE_DIR):
    raise FileNotFoundError(
        f"Transpilation directory not found: {TRANSPILE_DIR}\n"
        f"Run `find_best_transpilation_unrolled_boston_t1.ipynb` first to populate it."
    )

transpiled_per_exp = {}
transpile_summary_per_exp = {}

for exp in EXPERIMENTS:
    n = exp["n"]
    label = exp["label"]
    nd = os.path.join(TRANSPILE_DIR, f"n{n}")
    summary_file = os.path.join(nd, "summary.json")

    if not os.path.exists(summary_file):
        raise FileNotFoundError(
            f"Missing summary for N={n}: {summary_file}\n"
            f"Run `find_best_transpilation_unrolled_boston_t1.ipynb` for this N."
        )

    with open(summary_file) as f:
        n_summary = json.load(f)

    # Order paths the same way the strategy emits them: path_0, path_1, …
    golden_metadata = golden_data_per_exp[n]['metadata']
    path_order = [
        m.get('metadata', {}).get('path_name', f'path_{i}')
        for i, m in enumerate(golden_metadata)
    ]

    transpiled = []
    summary = []
    print(f"\n--- {label} ---")
    for path_name in path_order:
        if path_name not in n_summary.get('paths', {}):
            raise KeyError(
                f"Path {path_name!r} missing from {summary_file}. "
                f"Run the transpilation notebook to fill in this path."
            )
        path_meta = n_summary['paths'][path_name]
        qpy_file = os.path.join(nd, path_meta['qpy_file'])
        if not os.path.exists(qpy_file):
            raise FileNotFoundError(f"Missing QPY for {path_name}: {qpy_file}")

        with open(qpy_file, 'rb') as f:
            qcs = qpy.load(f)
        if len(qcs) != 1:
            raise ValueError(f"Expected 1 circuit in {qpy_file}, got {len(qcs)}")
        transpiled.append(qcs[0])

        summary.append({
            'path_name': path_name,
            'best_seed': path_meta['best_seed'],
            'best_2q_depth': path_meta['best_2q_depth'],
            'best_2q_gates': path_meta['best_2q_gates'],
            'depth': path_meta['depth'],
            'all_seed_2q_depths': [s['2q_depth'] for s in path_meta.get('all_seed_stats', [])],
        })
        print(f"  {path_name}: seed={path_meta['best_seed']}, 2Q depth={path_meta['best_2q_depth']}, "
              f"2Q gates={path_meta['best_2q_gates']}, gates={path_meta.get('gate_counts', {})}")

    transpiled_per_exp[n] = transpiled
    transpile_summary_per_exp[n] = summary

# Mirror the per-N summaries into the experiment results dir for traceability
combined_summary_path = os.path.join(RESULTS_DIR, "transpile_summary.json")
with open(combined_summary_path, 'w') as f:
    json.dump({str(n): transpile_summary_per_exp[n] for n in transpile_summary_per_exp}, f, indent=2)
print(f"\nTranspilation summary copy saved to: {combined_summary_path}")


--- N=3, K=2, T=1 ---
  path_0: seed=0, 2Q depth=18, 2Q gates=20, gates={'sx': 36, 'rz': 27, 'cz': 20, 'measure': 3, 'x': 1, 'reset': 1}
  path_1: seed=0, 2Q depth=18, 2Q gates=20, gates={'sx': 36, 'rz': 27, 'cz': 20, 'measure': 3, 'x': 1, 'reset': 1}

--- N=5, K=2, T=1 ---
  path_0: seed=0, 2Q depth=18, 2Q gates=40, gates={'sx': 72, 'rz': 54, 'cz': 40, 'measure': 4, 'x': 2, 'reset': 2}
  path_1: seed=0, 2Q depth=18, 2Q gates=40, gates={'sx': 72, 'rz': 54, 'cz': 40, 'measure': 4, 'x': 2, 'reset': 2}
  path_2: seed=0, 2Q depth=18, 2Q gates=40, gates={'sx': 72, 'rz': 54, 'cz': 40, 'measure': 4, 'x': 2, 'reset': 2}
  path_3: seed=0, 2Q depth=18, 2Q gates=40, gates={'sx': 72, 'rz': 54, 'cz': 40, 'measure': 4, 'x': 2, 'reset': 2}

--- N=7, K=2, T=1 ---
  path_0: seed=0, 2Q depth=18, 2Q gates=60, gates={'sx': 108, 'rz': 81, 'cz': 60, 'measure': 5, 'x': 4, 'reset': 3}
  path_1: seed=0, 2Q depth=18, 2Q gates=60, gates={'sx': 108, 'rz': 81, 'cz': 60, 'measure': 5, 'x': 4, 'reset': 3}
  path_2:

## Step 5: Noise Application & Circuit Instance Generation

Identical fast-path twirling as the full notebook: copy the transpiled circuit, sample Pauli noise, and inject ISA-safe gates at the front.

In [6]:
def build_noisy_batch(transpiled_golden, golden_circuits, golden_metadata, epsilon, n_random, batch_size):
    """Build batches of noisy circuit instances from pre-transpiled golden circuits."""
    all_batches = []
    batch_circuits = []
    batch_metadata = []

    for _ in range(n_random):
        noise_strategy = PauliTwirlingStrategy(K)

        for i, qc_transpiled in enumerate(transpiled_golden):
            qc_instance = qc_transpiled.copy()
            orig_qc = golden_circuits[i]

            data_regs = [reg for reg in orig_qc.qregs if reg.name.startswith("R")]
            noise_ops = noise_strategy.generate_noise_ops(data_regs, epsilon)

            layout = qc_transpiled.layout.initial_layout if qc_transpiled.layout else None

            for gate, logical_qubit in noise_ops:
                target_qubit = None
                if layout and logical_qubit in layout:
                    phys_qubit_idx = layout[logical_qubit]
                    target_qubit = qc_instance.qubits[phys_qubit_idx]
                elif logical_qubit in qc_instance.qubits:
                    target_qubit = logical_qubit
                else:
                    for q in qc_instance.qubits:
                        if hasattr(q, 'register') and hasattr(logical_qubit, 'register'):
                            if q.register.name == logical_qubit.register.name and q.index == logical_qubit.index:
                                target_qubit = q
                                break

                if target_qubit:
                    if gate.name == 'z':
                        qc_instance.data.insert(0, CircuitInstruction(RZGate(np.pi), (target_qubit,), ()))
                    elif gate.name == 'y':
                        qc_instance.data.insert(0, CircuitInstruction(RZGate(np.pi), (target_qubit,), ()))
                        qc_instance.data.insert(0, CircuitInstruction(XGate(), (target_qubit,), ()))
                    else:
                        qc_instance.data.insert(0, CircuitInstruction(gate, (target_qubit,), ()))

            batch_circuits.append(qc_instance)
            batch_metadata.append(golden_metadata[i])

            if len(batch_circuits) >= batch_size:
                all_batches.append((batch_circuits, batch_metadata))
                batch_circuits = []
                batch_metadata = []

    if batch_circuits:
        all_batches.append((batch_circuits, batch_metadata))

    return all_batches

print("build_noisy_batch() defined.")

build_noisy_batch() defined.


## Step 6: Submission with Incremental Checkpointing

After **every λ data point**, the partial result list is written to both `results_n{N}_k{K}_t{T}_unrolled_ibm_boston.csv` and the matching `.json`. If the kernel dies or the run is interrupted, re-running the cell will skip λs that already exist in the CSV and pick up where it left off.

In [7]:
def checkpoint_paths(n):
    base = f"results_n{n}_k{K}_t{T}_unrolled_ibm_boston"
    return (
        os.path.join(RESULTS_DIR, base + ".csv"),
        os.path.join(RESULTS_DIR, base + ".json"),
    )

def load_checkpoint(n):
    csv_path, _ = checkpoint_paths(n)
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        return df.to_dict('records')
    return []

def save_checkpoint(n, results, meta):
    csv_path, json_path = checkpoint_paths(n)
    df = pd.DataFrame(results)
    df.to_csv(csv_path, index=False)
    with open(json_path, 'w') as f:
        json.dump({'meta': meta, 'results': results}, f, indent=2)

def run_experiment(n, transpiled_golden, golden_circuits, golden_metadata,
                   lambdas, n_random, shots_per_circuit, batch_size, backend):
    """λ sweep for one N. Resumes from CSV checkpoint if it exists."""
    result_processor = ResultProcessor(K)
    sampler = IBMSampler(mode=backend)

    results = load_checkpoint(n)
    done_lambdas = {round(float(r['lambda']), 4) for r in results}
    if done_lambdas:
        print(f"  Resuming N={n}: {len(done_lambdas)} λ point(s) already in checkpoint -> {sorted(done_lambdas)}")

    meta = {
        'n': n, 'k': K, 't': T,
        'n_random': n_random, 'shots_per_circuit': shots_per_circuit,
        'batch_size': batch_size, 'backend': backend.name,
        'transpile_seeds': TRANSPILE_SEEDS, 'opt_level': OPT_LEVEL,
    }

    for epsilon in tqdm(lambdas, desc=f"N={n} λ sweep"):
        eps_key = round(float(epsilon), 4)
        if eps_key in done_lambdas:
            continue

        batches = build_noisy_batch(
            transpiled_golden, golden_circuits, golden_metadata,
            epsilon, n_random, batch_size,
        )

        global_path_stats = defaultdict(lambda: {'success': 0, 'total': 0})

        for batch_idx, (batch_circs, batch_meta) in enumerate(batches):
            pubs = [(qc, None, shots_per_circuit) for qc in batch_circs]
            try:
                job = sampler.run(pubs)
                pub_result = job.result()

                extracted_counts = ResultProcessor.extract_counts_from_job_result(pub_result, is_dynamic=False)

                total_clbits_list = []
                for counts in extracted_counts:
                    if counts:
                        first_key = next(iter(counts))
                        total_clbits_list.append(len(first_key.replace(" ", "")))
                    else:
                        total_clbits_list.append(0)

                batch_stats = result_processor.aggregate_batch_stats(
                    extracted_counts, batch_meta, total_clbits_list
                )
                for cond_key, stats in batch_stats.items():
                    global_path_stats[cond_key]['success'] += stats['success']
                    global_path_stats[cond_key]['total'] += stats['total']

            except Exception as e:
                print(f"  Error at λ={epsilon:.4f}, batch {batch_idx}: {e}")
                import traceback
                traceback.print_exc()

        fidelity = 0.0
        for stats in global_path_stats.values():
            if stats['total'] > 0:
                fidelity += stats['success'] / stats['total']

        results.append({'lambda': float(epsilon), 'fidelity': float(fidelity)})
        results.sort(key=lambda r: r['lambda'])
        save_checkpoint(n, results, meta)
        done_lambdas.add(eps_key)
        print(f"  N={n}, λ={epsilon:.4f} -> fidelity={fidelity:.4f}  [checkpoint saved]")

    return results

print("run_experiment() defined.")

run_experiment() defined.


### Run All Experiments (N ∈ {3, 5, 7, 9, 11})

Loops over every N. Each N is checkpointed independently — if the kernel dies, re-run the cell to resume.

In [8]:
results_per_exp = {}
df_per_exp = {}

for exp in EXPERIMENTS:
    n = exp["n"]
    print(f"\n========== {exp['label']} ==========")
    results_n = run_experiment(
        n=n,
        transpiled_golden=transpiled_per_exp[n],
        golden_circuits=golden_data_per_exp[n]['circuits'],
        golden_metadata=golden_data_per_exp[n]['metadata'],
        lambdas=lambdas,
        n_random=N_RANDOM,
        shots_per_circuit=SHOTS,
        batch_size=BATCH_SIZE,
        backend=backend,
    )
    results_per_exp[n] = results_n
    df_per_exp[n] = pd.DataFrame(results_n)
    print(f"N={n}: {len(df_per_exp[n])} λ points collected")

for n in sorted(df_per_exp):
    print(f"\n--- N={n} ---")
    print(df_per_exp[n].to_string(index=False))


========== N=3, K=2, T=1 ==========


N=3 λ sweep:   0%|          | 0/10 [00:00<?, ?it/s]

  N=3, λ=0.0000 -> fidelity=0.9785  [checkpoint saved]
  N=3, λ=0.1000 -> fidelity=0.9430  [checkpoint saved]
  N=3, λ=0.2000 -> fidelity=0.8837  [checkpoint saved]
  N=3, λ=0.3000 -> fidelity=0.8245  [checkpoint saved]
  N=3, λ=0.4000 -> fidelity=0.7383  [checkpoint saved]
  N=3, λ=0.5000 -> fidelity=0.6783  [checkpoint saved]
  N=3, λ=0.6000 -> fidelity=0.5847  [checkpoint saved]
  N=3, λ=0.7000 -> fidelity=0.5113  [checkpoint saved]
  N=3, λ=0.8000 -> fidelity=0.4285  [checkpoint saved]
  N=3, λ=0.9000 -> fidelity=0.3403  [checkpoint saved]
N=3: 10 λ points collected

========== N=5, K=2, T=1 ==========


N=5 λ sweep:   0%|          | 0/10 [00:00<?, ?it/s]

  N=5, λ=0.0000 -> fidelity=0.9691  [checkpoint saved]
  N=5, λ=0.1000 -> fidelity=0.9463  [checkpoint saved]
  N=5, λ=0.2000 -> fidelity=0.8987  [checkpoint saved]
  N=5, λ=0.3000 -> fidelity=0.8451  [checkpoint saved]
  N=5, λ=0.4000 -> fidelity=0.7765  [checkpoint saved]
  N=5, λ=0.5000 -> fidelity=0.7023  [checkpoint saved]
  N=5, λ=0.6000 -> fidelity=0.6034  [checkpoint saved]
  N=5, λ=0.7000 -> fidelity=0.5359  [checkpoint saved]
  N=5, λ=0.8000 -> fidelity=0.4393  [checkpoint saved]
  N=5, λ=0.9000 -> fidelity=0.3357  [checkpoint saved]
N=5: 10 λ points collected

========== N=7, K=2, T=1 ==========


...

N=9 λ sweep:   0%|          | 0/10 [00:00<?, ?it/s]

  Error at λ=0.0000, batch 0: 'Failed to run program: "(\'Connection aborted.\', ConnectionResetError(54, \'Connection reset by peer\'))"'


Traceback (most recent call last):
  File "/var/folders/jt/44gl1rhd26v6ykt0frqzl87r0000gn/T/ipykernel_87890/1090284899.py", line 55, in run_experiment
    job = sampler.run(pubs)
  File "/Users/henryzou/.venvs/opqa/lib/python3.13/site-packages/qiskit_ibm_runtime/sampler.py", line 111, in run
    return self._run(coerced_pubs)
           ~~~~~~~~~^^^^^^^^^^^^^^
  File "/Users/henryzou/.venvs/opqa/lib/python3.13/site-packages/qiskit_ibm_runtime/base_primitive.py", line 181, in _run
    return self._service._run(
           ~~~~~~~~~~~~~~~~~~^
        program_id=self._program_id(),
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
        calibration_id=calibration_id,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/henryzou/.venvs/opqa/lib/python3.13/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py", line 1025, in _run
    raise IBMRuntimeError(f"Failed to run program: {ex}") from None
qiskit_ibm_runtime.exceptions.IBMRuntimeError: 'Failed to run prog

  N=9, λ=0.0000 -> fidelity=0.8390  [checkpoint saved]


KeyboardInterrupt: 

## Step 7: Plot — Experimental vs. Theory

Theory curves for K=2 (d=4):
- N=3: $F(\lambda) = \tfrac{1}{8}(8 - 2\lambda - 7\lambda^2 + 3\lambda^3)$
- N=5: $F(\lambda) = \tfrac{1}{640}(640 - 96\lambda - 224\lambda^2 - 700\lambda^3 + 693\lambda^4 - 153\lambda^5)$
- N=7: $F(\lambda) = \tfrac{1}{215040}(215040 - 23040\lambda - 50688\lambda^2 - 88768\lambda^3 - 311056\lambda^4 + 497192\lambda^5 - 207951\lambda^6 + 23031\lambda^7)$

In [ ]:
def theory_curve(lam, n, k):
    """Closed-form theory for K=2, available only for N=3,5,7. Returns None otherwise."""
    if k == 2:
        if n == 3:
            return (1/8) * (8 - 2*lam - 7*lam**2 + 3*lam**3)
        elif n == 5:
            return (1/640) * (640 - 96*lam - 224*lam**2 - 700*lam**3 + 693*lam**4 - 153*lam**5)
        elif n == 7:
            return (1/215040) * (215040 - 23040*lam - 50688*lam**2 - 88768*lam**3
                                  - 311056*lam**4 + 497192*lam**5 - 207951*lam**6 + 23031*lam**7)
    return None

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 16,
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 10,
    'lines.linewidth': 2, 'lines.markersize': 7,
})

fig, ax = plt.subplots(figsize=(13, 8))
lam_fine = np.linspace(0, 1, 200)

n_values_sorted = sorted(df_per_exp.keys())
cmap = plt.get_cmap('viridis')
colors = {n: cmap(i / max(1, len(n_values_sorted) - 1)) for i, n in enumerate(n_values_sorted)}
marker_cycle = ['o', '^', 's', 'D', 'v', 'P', 'X', '*', 'h', '<']
markers = {n: marker_cycle[i % len(marker_cycle)] for i, n in enumerate(n_values_sorted)}

for n in n_values_sorted:
    df = df_per_exp[n]
    th = theory_curve(lam_fine, n, K)
    if th is not None:
        ax.plot(lam_fine, th, '--', color=colors[n], linewidth=1.3, alpha=0.55,
                label=f'Theory (N={n}, K=2)')
    if len(df) > 0:
        ax.scatter(df['lambda'], df['fidelity'], color=colors[n], marker=markers[n], s=60,
                   zorder=5, alpha=0.9, edgecolors='white', linewidth=0.5,
                   label=f'IBM Boston (N={n}, K=2, T=1)')

ax.axhline(y=0.25, color='gray', linestyle=':', alpha=0.4, label='Random guess (1/d = 0.25)')
ax.set_title('QPA Fidelity Decay — Unrolled (T=1) — IBM Boston', fontweight='bold')
ax.set_xlabel(r'Depolarizing Noise Strength ($\lambda$)')
ax.set_ylabel('Purified Fidelity')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0.20, 1.05)
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), frameon=True, shadow=True, ncol=1)
ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
plt.tight_layout()

plot_path = os.path.join(RESULTS_DIR, "fidelity_decay_unrolled_t1_ibm_boston.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"Plot saved to: {plot_path}")
plt.show()

## Step 8: Result Inspection

In [ ]:
for n in sorted(df_per_exp):
    df = df_per_exp[n]
    print(f"\n=== N={n}, K={K}, T={T} ===")
    print(f"{'Lambda':>8} {'Exp Fidelity':>14} {'Theory':>12} {'Delta':>10}")
    print("-" * 48)
    for _, row in df.iterrows():
        lam = row['lambda']
        exp_f = row['fidelity']
        th_f = theory_curve(lam, n, K)
        if th_f is not None:
            delta = exp_f - th_f
            print(f"{lam:8.4f} {exp_f:14.4f} {th_f:12.4f} {delta:+10.4f}")
        else:
            print(f"{lam:8.4f} {exp_f:14.4f} {'n/a':>12} {'n/a':>10}")

## Step 9: Final Consolidated Save

Per-N CSV/JSON files are already written incrementally during the sweep. This step combines them into a single consolidated CSV + JSON for convenience.

In [ ]:
all_runs = dict(df_per_exp)

combined_rows = []
for n in sorted(all_runs):
    df = all_runs[n]
    for _, row in df.iterrows():
        lam = float(row['lambda'])
        exp_f = float(row['fidelity'])
        th_f = theory_curve(lam, n, K)
        combined_rows.append({
            'n': n, 'k': K, 't': T,
            'lambda': lam,
            'fidelity_experiment': exp_f,
            'fidelity_theory': float(th_f) if th_f is not None else None,
            'delta': (exp_f - float(th_f)) if th_f is not None else None,
        })

combined_df = pd.DataFrame(combined_rows)

combined_csv = os.path.join(RESULTS_DIR, "results_all_unrolled_ibm_boston.csv")
combined_json = os.path.join(RESULTS_DIR, "results_all_unrolled_ibm_boston.json")

combined_df.to_csv(combined_csv, index=False)

run_meta = {
    'backend': backend.name,
    'k': K, 't': T,
    'n_values': sorted(all_runs.keys()),
    'lambdas': lambdas.tolist(),
    'n_random': N_RANDOM,
    'shots_per_circuit': SHOTS,
    'batch_size': BATCH_SIZE,
    'transpile_seeds': TRANSPILE_SEEDS,
    'opt_level': OPT_LEVEL,
}
with open(combined_json, 'w') as f:
    json.dump({'meta': run_meta, 'results': combined_rows}, f, indent=2)

print(f"Combined CSV  -> {combined_csv}")
print(f"Combined JSON -> {combined_json}")
combined_df